# **Data preprocessing**

## Dataset

**GoEmotions**: Multiclass classification dataset, extracted from Reddit by Google. It features 27 emotional labels, plus a "neutral" label.

Dataset source: https://www.kaggle.com/datasets/debarshichanda/goemotions

---

## Dataset Details

In [1]:
# Import neccessary libraries

import pandas as pd
import numpy as np

In [2]:
# Read data files and built dataframe
df1 = pd.read_csv('../data/goemotions_1.csv')
df2 = pd.read_csv('../data/goemotions_2.csv')
df3 = pd.read_csv('../data/goemotions_3.csv')
df = pd.concat([df1, df2, df3], ignore_index=True)

display(df.head())
print(f"Total number of rows: {len(df)}")

,text,id,author,subreddit,link_id,parent_id,created_utc,rater_id,example_very_unclear,admiration,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,That game hurt.,eew5j0j,Brdd9,nrl,t3_ajis4z,t1_eew18eq,1.548381e+09,1,False,0,...,0,0,0,0,0,0,0,1,0,0
1,>sexuality shouldn’t be a grouping category I...,eemcysk,TheGreen888,unpopularopinion,t3_ai4q37,t3_ai4q37,1.548084e+09,37,True,0,...,0,0,0,0,0,0,0,0,0,0
2,"You do right, if you don't care then fuck 'em!",ed2mah1,Labalool,confessions,t3_abru74,t1_ed2m7g7,1.546428e+09,37,False,0,...,0,0,0,0,0,0,0,0,0,1
3,Man I love reddit.,eeibobj,MrsRobertshaw,facepalm,t3_ahulml,t3_ahulml,1.547965e+09,18,False,0,...,1,0,0,0,0,0,0,0,0,0
4,"[NAME] was nowhere near them, he was by the Fa...",eda6yn6,American_Fascist713,starwarsspeculation,t3_ackt2f,t1_eda65q2,1.546669e+09,2,False,0,...,0,0,0,0,0,0,0,0,0,1


Total number of rows: 211225


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211225 entries, 0 to 211224
Data columns (total 37 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   text                  211225 non-null  object 
 1   id                    211225 non-null  object 
 2   author                211225 non-null  object 
 3   subreddit             211225 non-null  object 
 4   link_id               211225 non-null  object 
 5   parent_id             211225 non-null  object 
 6   created_utc           211225 non-null  float64
 7   rater_id              211225 non-null  int64  
 8   example_very_unclear  211225 non-null  bool   
 9   admiration            211225 non-null  int64  
 10  amusement             211225 non-null  int64  
 11  anger                 211225 non-null  int64  
 12  annoyance             211225 non-null  int64  
 13  approval              211225 non-null  int64  
 14  caring                211225 non-null  int64  
 15  

The dataset features the following columns:
- `text`: The text of the Reddit comment.
- `id`: The unique id of the comment.
- `author`: Username of the Redditor who wrote the comment.
- `subreddit`: The subreddit containing the thread in which the comment was written.
- `link_id`: The link id of the comment.
- `parent_id`: The parent id of the comment.
- `created_utc`: The timestamp of the comment.
- `rater_id`: The unique id of the human annotator who classified the comment.
- `example_very_unclear`: Whether the annotator marked the example as being very unclear or difficult to label, leading to no classification label selected.
- Separate columns representing each of the emotion categories, with binary labels (0 or 1). Featured emotion categories are: `admiration`, `amusement`, `anger`, `annoyance`, `approval`, `caring`, `confusion`, `curiosity`, `desire`, `disappointment`, `disapproval`, `disgust`, `embarrassment`, `excitement`, `fear`, `gratitude`, `grief`, `joy`, `love`, `nervousness`, `optimism`, `pride`, `realization`, `relief`, `remorse`, `sadness`, `surprise`, and `neutral`.

---

## Process duplicated rows

In [4]:
# Check for duplicated rows
is_duplicated = df.duplicated()
if is_duplicated.any():
    df.drop_duplicates(inplace=True)
    print(is_duplicated.sum(), ' duplicated rows found and removed')
else: 
    print('No duplicated rows found')

No duplicated rows found



---

## Process missing values

In [5]:
# Check for missing values
missing_values = df.isnull().sum()
if missing_values.any():
    print('Missing values found:')
    print(missing_values[missing_values > 0])
else:
    print('No missing values found')

No missing values found



---

## Handling unclear exmaples

In [6]:
# Percentage of objects with example_very_unclear=True
unclear_percentage = df['example_very_unclear'].mean() * 100
print(f"Percentage of objects with example_very_unclear=True: {unclear_percentage:.2f}%")

Percentage of objects with example_very_unclear=True: 1.61%


Less than $2\%$ of rows are labeled as unclear. These are intentionally marked by annotators as ambiguous or not clearly labeled, and thus always have all emotion labels set to 0. Including these rows into training might confuse the model, so I've decided to simply drop them.

In [7]:
# Drop rows with unclear examples
df = df[df['example_very_unclear'] == False]
print("Rows with unclear examples dropped.")
print("New total number of rows:", len(df))

Rows with unclear examples dropped.
New total number of rows: 207814



---

## Extract Useful Columns

Since everything other than the text of the comment and the detonation values are just metadata, which are unused in the training of the model, I will extract the meaningful columns from the dataset and save them into a separate file.

In [8]:
# Extract meaningful columns
useful_columns = ['text', 'admiration', 'amusement', 'anger', 'annoyance',
                  'approval', 'caring', 'confusion', 'curiosity', 'desire',
                  'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement',
                  'fear', 'gratitude', 'grief', 'joy', 'love', 
                  'nervousness', 'optimism', 'pride', 'realization', 'relief', 
                  'remorse', 'sadness', 'surprise', 'neutral']
df = df[useful_columns]
print(f"Final dataset size: {df.shape}")

# Save the processed dataframe to a CSV file
df.to_csv('../data/processed_data.csv', index=False)
print("Processed data saved to '../data/processed_data.csv'")

Final dataset size: (207814, 29)
Processed data saved to '../data/processed_data.csv'
